# Il prodotto scalare: la domanda "quanto siete d'accordo?"

*Anatomia di un LLM* · capitolo 3, I mattoni · script `01-02`, `scripts/01_02_prodotto_scalare.py`

Due liste di numeri della stessa lunghezza si moltiplicano posizione per
posizione e si sommano. Esce un numero solo, ed e' grande quando le due liste
puntano nella stessa direzione. E' l'unica operazione che serve capire per
capire l'attenzione, e vale la pena passarci qualche minuto adesso.

Si esegue con `python3 scripts/01_02_prodotto_scalare.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/01_02_prodotto_scalare.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
import torch
import torch.nn.functional as F

from llm_anatomy.utils import prepara, titolo, verifica

prepara()

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


device(type='cpu')

In [3]:
titolo("Quattro coppie, quattro risposte")
a = torch.tensor([1.0, 0.0])          # una freccia che punta a destra
casi = {
    "stessa direzione": torch.tensor([2.0, 0.0]),
    "ad angolo retto": torch.tensor([0.0, 3.0]),
    "direzione opposta": torch.tensor([-2.0, 0.0]),
    "a quarantacinque gradi": torch.tensor([1.0, 1.0]),
}
print(f"  la prima freccia e' sempre {a.tolist()}, cioe' punta a destra\n")
print(f"  {'la seconda':<24} {'':<12} {'a.b':>7} {'gradi':>7}")
for nome, b in casi.items():
    coseno = F.cosine_similarity(a, b, dim=0).clamp(-1, 1)
    gradi = torch.rad2deg(torch.acos(coseno))
    print(f"  {nome:<24} {str(b.tolist()):<12} "
          f"{torch.dot(a, b):7.2f} {gradi:7.0f}")
verifica(torch.dot(a, casi["ad angolo retto"]).item() == 0.0,
         "due direzioni ad angolo retto danno esattamente zero")
print("  positivo se sono d'accordo, zero se non c'entrano nulla,")
print("  negativo se si contraddicono")


Quattro coppie, quattro risposte
--------------------------------
  la prima freccia e' sempre [1.0, 0.0], cioe' punta a destra

  la seconda                                a.b   gradi
  stessa direzione         [2.0, 0.0]      2.00       0
  ad angolo retto          [0.0, 3.0]      0.00      90
  direzione opposta        [-2.0, 0.0]    -2.00     180
  a quarantacinque gradi   [1.0, 1.0]      1.00      45
  [verifica superata] due direzioni ad angolo retto danno esattamente zero
  positivo se sono d'accordo, zero se non c'entrano nulla,
  negativo se si contraddicono


In [4]:
titolo("Attenzione pero': misura anche la lunghezza")
lungo = torch.tensor([10.0, 10.0])
corto = torch.tensor([0.1, 0.1])
for nome, b in (("lunga", lungo), ("corta", corto)):
    coseno = F.cosine_similarity(a, b, dim=0)
    print(f"  freccia {nome:<6} lunghezza {b.norm():6.2f}   "
          f"a.b {torch.dot(a, b):7.2f}   angolo uguale (coseno {coseno:.3f})")
print("  stessa direzione, prodotti scalari lontanissimi: il numero cresce")
print("  se le frecce si allungano, anche quando l'accordo non cambia")


Attenzione pero': misura anche la lunghezza
-------------------------------------------
  freccia lunga  lunghezza  14.14   a.b   10.00   angolo uguale (coseno 0.707)
  freccia corta  lunghezza   0.14   a.b    0.10   angolo uguale (coseno 0.707)
  stessa direzione, prodotti scalari lontanissimi: il numero cresce
  se le frecce si allungano, anche quando l'accordo non cambia


In [5]:
titolo("Piu' numeri, punteggi piu' grandi")
print(f"  {'lunghezza':>10} {'media':>9} {'quanto oscilla':>16} {'radice':>9}")
for dim in (8, 32, 128, 512):
    x = torch.randn(4096, dim)
    y = torch.randn(4096, dim)
    p = (x * y).sum(dim=1)
    print(f"  {dim:10d} {p.mean():9.3f} {p.std():16.3f} {dim ** 0.5:9.3f}")
print("  la media resta zero, ma l'oscillazione cresce come la radice della")
print("  lunghezza: sommare 512 termini invece di 8 da' numeri piu' grandi")
print("  anche se ogni termine e' rimasto della stessa taglia")


Piu' numeri, punteggi piu' grandi
---------------------------------
   lunghezza     media   quanto oscilla    radice
           8    -0.054            2.771     2.828
          32     0.124            5.660     5.657
         128    -0.199           11.208    11.314
         512     0.103           22.717    22.627
  la media resta zero, ma l'oscillazione cresce come la radice della
  lunghezza: sommare 512 termini invece di 8 da' numeri piu' grandi
  anche se ogni termine e' rimasto della stessa taglia


## Da provare

porta d a 4096 e guarda l'ultima tabella: l'oscillazione arriva a 64.